### np.linalg.eig

In [3]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _get_item(a, idx):
    cur = a
    for i in idx:
        cur = cur[i]
    return cur


def _build_nested(shape, getter, prefix=()):
    if len(shape) == 0:
        return getter(prefix)
    return [_build_nested(shape[1:], getter, prefix + (i,)) for i in range(shape[0])] 


In [4]:
import math
import cmath

def _det_2x2(mat):
    return mat[0][0] * mat[1][1] - mat[0][1] * mat[1][0]


def _trace_2x2(mat):
    return mat[0][0] + mat[1][1]


def _eig_2x2_analytic(mat):
    """
    Returns (eigenvalues, eigenvectors).
    eigenvalues: list of 2 numbers (float or complex‑like)
    eigenvectors: list of 2 lists, each of length 2.
    """
    a, b = mat[0]
    c, d = mat[1]
    tr = a + d
    det = a * d - b * c

    # cubic in reals
    discriminant = tr * tr - 4 * det
    if discriminant >= 0:
        lamb1 = (tr + discriminant**0.5) * 0.5
        lamb2 = (tr - discriminant**0.5) * 0.5
    else:
        lamb1 = (tr + 1j * (-discriminant)**0.5) * 0.5
        lamb2 = (tr - 1j * (-discriminant)**0.5) * 0.5

    def _make_eigenvector(lambd):
        # solve (A - λI) v = 0
        # [[a-λ, b], [c, d-λ]] v = 0
        da = a - lambd
        db = b
        dc = c
        dd = d - lambd
        # try to pick a non‑trivial solution
        if db != 0:
            # set v[1] = 1, solve for v[0]
            return [-(dd) / db if db else 0, 1.0]
        elif da != 0:
            return [1.0, -da / dc if dc else 0]
        else:
            return [1.0, 1.0]

    try:
        e1 = _make_eigenvector(lamb1)
        e2 = _make_eigenvector(lamb2)
        return [lamb1, lamb2], [e1, e2]
    except Exception:
        # fallback: simple choices
        return [lamb1, lamb2], [[1, 0], [0, 1]]

def np_linalg_eig(a):
    """
    Compute eigenvalues and right eigenvectors of a square matrix.
    For now, supports only 2×2 matrices.
    Returns:
        eigenvalues: list of eigenvalues
        eigenvectors: list of lists, where each inner list is an eigenvector (columns)
    """
    shape = get_shape(a)
    if len(shape) != 2:
        raise ValueError("Matrix must be 2-D")
    if shape[0] != shape[1]:
        raise ValueError("Matrix must be square")

    n = shape[0]
    if n == 2:
        return _eig_2x2_analytic(a)

    # 3×3 and more would need full numeric solvers (e.g., power method, QR, etc.).
    # For now, raise NotImplementedError for bigger matrices.
    raise NotImplementedError("Only 2×2 matrices supported in this version")

In [5]:
print(np_linalg_eig([[1, 0], [0, 1]]))
print(np_linalg_eig([[2, 0], [0, 3]]))      
print(np_linalg_eig([[0, 1], [1, 0]]))      
print(np_linalg_eig([[1, 1], [0, 1]]))      
print(np_linalg_eig([[4, -2], [1, 1]]))     
print(np_linalg_eig([[1, 2], [2, 1]]))      

([1.0, 1.0], [[1.0, 1.0], [1.0, 1.0]])
([3.0, 2.0], [[1.0, 0], [1.0, 1.0]])
([1.0, -1.0], [[1.0, 1.0], [-1.0, 1.0]])
([1.0, 1.0], [[-0.0, 1.0], [-0.0, 1.0]])
([3.0, 2.0], [[-1.0, 1.0], [-0.5, 1.0]])
([3.0, -1.0], [[1.0, 1.0], [-1.0, 1.0]])


In [6]:
print(np_linalg_eig([[1, 2], [3]])) 

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [7]:
print(np_linalg_eig([[1, 2, 3], [4, 5, 6]]))  

ValueError: Matrix must be square

In [8]:
print(np_linalg_eig([[1, 2], [3, 4], [5, 6]]))

ValueError: Matrix must be square

In [9]:
print(np_linalg_eig(42))   

ValueError: Matrix must be 2-D